# Generación del conjunto sintético binario para \(L=200\) — versión comparable con \(L=100\)

Este cuaderno genera los ficheros HDF5 necesarios para entrenar y evaluar **ConvTransformer** con trayectorias de longitud \(L=200\).

Mejoras incorporadas antes del entrenamiento global:

- se añade ruido de localización con `NOISE_LEVELS = [0.1, 0.5, 1.0]`, como en el protocolo \(L=100\);
- el punto de cambio se muestrea de forma relativa, con \(u \in [0.2,0.8]\) y `cp = round(u * LENGTH)`;
- para \(L=200\), esto equivale a `cp` entre 40 y 160, manteniendo la misma posición relativa que en \(L=100\);
- se guardan `noise_sigma`, `cp_relative_position` y todos los metadatos necesarios para revisar el conjunto.

Los ficheros generados son compatibles con los notebooks de entrenamiento del Protocolo B.


## 1. Instalación opcional

Ejecuta esta celda solo si tu entorno no tiene instalado `andi_datasets`.

> Para la generación final del TFM se recomienda usar `andi_datasets`, ya que es la librería asociada al contexto AnDi.

In [ ]:
# Descomenta esta línea si necesitas instalar la librería en Colab o en tu entorno.
# !pip install -q andi-datasets

## 2. Importación de librerías y configuración general

Por defecto el cuaderno está en modo `FAST`, para comprobar rápidamente que todo funciona.  
Cuando la generación rápida salga bien, cambia:

```python
RUN_MODE = "GLOBAL"
```

para generar los ficheros completos.

In [ ]:
import json
import os
import math
import random
from pathlib import Path
from datetime import datetime
from importlib.metadata import PackageNotFoundError, version

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
try:
    ANDI_DATASETS_VERSION = version("andi-datasets")
except PackageNotFoundError:
    ANDI_DATASETS_VERSION = "not-installed"

# -------------------------------------------------------------------
# Modo de ejecución
# -------------------------------------------------------------------
# Usa "FAST" para validar el flujo de trabajo.
# Usa "GLOBAL" para generar los datos definitivos.
RUN_MODE = os.environ.get("TFM_RUN_MODE", "FAST").upper()
OVERWRITE_EXISTING = os.environ.get("TFM_OVERWRITE", "0") == "1"
if RUN_MODE not in {"FAST", "GLOBAL"}:
    raise ValueError("TFM_RUN_MODE must be FAST or GLOBAL")

# -------------------------------------------------------------------
# Parámetros principales del problema
# -------------------------------------------------------------------
LENGTH = 200
DIMENSION = 1

# Para conservar la misma posición relativa que en L=100 (20%–80%).
CP_RELATIVE_MIN = 0.20
CP_RELATIVE_MAX = 0.80
MIN_SEGMENT_LENGTH = int(round(CP_RELATIVE_MIN * LENGTH))  # 40 para L=200

VALID_CP_MIN = MIN_SEGMENT_LENGTH
VALID_CP_MAX = LENGTH - MIN_SEGMENT_LENGTH

DX_LENGTH = LENGTH - 1
VALID_DX_MIN = VALID_CP_MIN - 1
VALID_DX_MAX = VALID_CP_MAX - 1

# Mismos niveles de ruido que en el protocolo L=100.
NOISE_LEVELS = [0.1, 0.5, 1.0]

MODEL_NAMES = ["ATTM", "CTRW", "FBM", "LW", "SBM"]
MODEL_TO_ID = {name: i for i, name in enumerate(MODEL_NAMES)}
TRANSITIONS = [(m1, m2) for m1 in MODEL_NAMES for m2 in MODEL_NAMES if m1 != m2]

# -------------------------------------------------------------------
# Tamaños de los conjuntos
# -------------------------------------------------------------------
FAST_SPLIT_SIZES = {
    "train": 20_000,
    "val": 4_000,
    "test": 10_000,
}

GLOBAL_SPLIT_SIZES = {
    "train": 200_000,
    "val": 20_000,
    "test": 200_000,
}

SPLIT_SIZES = FAST_SPLIT_SIZES if RUN_MODE.upper() == "FAST" else GLOBAL_SPLIT_SIZES

# -------------------------------------------------------------------
# Directorios
# -------------------------------------------------------------------
DEFAULT_PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROJECT_ROOT = Path(os.environ.get("TFM_PROJECT_ROOT", str(DEFAULT_PROJECT_ROOT))).expanduser()
DATA_DIR = PROJECT_ROOT / "data_synthetic_with_without_changepoint_dx"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("RUN_MODE:", RUN_MODE)
print("LENGTH:", LENGTH)
print("DX_LENGTH:", DX_LENGTH)
print("CP relative range:", (CP_RELATIVE_MIN, CP_RELATIVE_MAX))
print("VALID_CP:", (VALID_CP_MIN, VALID_CP_MAX))
print("VALID_DX:", (VALID_DX_MIN, VALID_DX_MAX))
print("NOISE_LEVELS:", NOISE_LEVELS)
print("DATA_DIR:", DATA_DIR)
print("SPLIT_SIZES:", SPLIT_SIZES)


## 3. Rangos de \(\alpha\)

Se muestrea un exponente anómalo \(\alpha\) para cada segmento.  
Los rangos respetan la naturaleza general de los modelos:

- CTRW y ATTM: subdifusivos;
- LW: superdifusivo;
- FBM y SBM: rango amplio.

In [ ]:
ALPHA_RANGES = {
    "ATTM": (0.05, 1.00),
    "CTRW": (0.05, 1.00),
    "FBM":  (0.05, 1.95),
    "LW":   (1.05, 1.95),
    "SBM":  (0.05, 1.95),
}

def sample_alpha(model_name, rng):
    low, high = ALPHA_RANGES[model_name]
    return float(rng.uniform(low, high))

## 4. Generadores de trayectorias

La función principal es `generate_model_trajectory`.

Primero intenta usar `andi_datasets`.  
Si la API instalada en tu entorno no coincide exactamente, el cuaderno muestra un error claro.

> Nota: se incluye un generador aproximado de respaldo solo para depuración estructural. Para los resultados finales del TFM se debe usar `andi_datasets`.

In [ ]:
ALLOW_FALLBACK_GENERATORS = False  # Cambiar a True solo para probar la estructura sin andi_datasets

def import_andi_models():
    """
    Intenta importar la clase models_theory de andi_datasets.
    """
    try:
        from andi_datasets.models_theory import models_theory
        return models_theory()
    except Exception as exc:
        print("No se pudo importar andi_datasets.models_theory.models_theory.")
        print("Error:", repr(exc))
        return None

ANDI_MODEL_OBJECT = import_andi_models()

if ANDI_MODEL_OBJECT is not None:
    print("Objeto de modelos AnDi cargado correctamente.")
    print("Métodos disponibles que no empiezan por '_':")
    print([m for m in dir(ANDI_MODEL_OBJECT) if not m.startswith("_")][:50])
else:
    if not ALLOW_FALLBACK_GENERATORS:
        print("Instala andi_datasets o activa ALLOW_FALLBACK_GENERATORS=True solo para depuración.")

In [ ]:
def _as_1d_track(x, desired_length):
    """
    Convierte la salida del generador en una trayectoria 1D de longitud deseada.
    """
    # Algunos generadores devuelven tuplas/listas con trayectoria y etiquetas.
    if isinstance(x, (tuple, list)):
        x = x[0]

    arr = np.asarray(x, dtype=np.float32)
    arr = np.squeeze(arr)

    # Si viene en formato (T, dim) o (dim, T), quedarse con la dimensión 1D.
    if arr.ndim == 2:
        if arr.shape[0] == desired_length:
            arr = arr[:, 0]
        elif arr.shape[1] == desired_length:
            arr = arr[0, :]
        else:
            arr = arr.reshape(-1)

    arr = arr.reshape(-1)

    if len(arr) < desired_length:
        raise ValueError(f"La trayectoria generada tiene longitud {len(arr)}, menor que {desired_length}.")

    arr = arr[:desired_length].astype(np.float32)
    arr = arr - arr[0]
    return arr


def _try_call_generator(method, T, alpha):
    """
    Prueba varias firmas habituales de los métodos de andi_datasets.
    """
    attempts = [
        lambda: method(T=T, alpha=alpha),
        lambda: method(T=T, alpha=alpha, dimension=1),
        lambda: method(T=T, alpha=alpha, dimensions=1),
        lambda: method(T=T, alpha=alpha, D=1),
        lambda: method(T, alpha),
        lambda: method(T, alpha, 1),
    ]

    last_error = None
    for attempt in attempts:
        try:
            return attempt()
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f"No se pudo llamar al generador con las firmas probadas. Último error: {last_error}")


def generate_with_andi(model_name, T, alpha):
    """
    Genera una trayectoria usando andi_datasets.models_theory si está disponible.
    """
    if ANDI_MODEL_OBJECT is None:
        raise RuntimeError("andi_datasets no está disponible.")

    candidate_method_names = [
        model_name.lower(),
        model_name.upper(),
        model_name,
    ]

    for method_name in candidate_method_names:
        if hasattr(ANDI_MODEL_OBJECT, method_name):
            method = getattr(ANDI_MODEL_OBJECT, method_name)
            raw = _try_call_generator(method, T, alpha)
            return _as_1d_track(raw, T)

    raise AttributeError(
        f"No se encontró un método para {model_name} en models_theory. "
        f"Métodos disponibles: {[m for m in dir(ANDI_MODEL_OBJECT) if not m.startswith('_')][:80]}"
    )


# -------------------------------------------------------------------
# Generadores aproximados de respaldo: SOLO depuración, no resultados finales.
# -------------------------------------------------------------------
def _fallback_fbm(T, alpha, rng):
    H = np.clip(alpha / 2.0, 0.03, 0.97)
    n = T - 1
    k = np.arange(n)
    gamma = 0.5 * (
        np.abs(k + 1) ** (2 * H)
        - 2 * np.abs(k) ** (2 * H)
        + np.abs(k - 1) ** (2 * H)
    )
    cov = np.empty((n, n), dtype=np.float64)
    for i in range(n):
        cov[i] = gamma[np.abs(np.arange(n) - i)]
    cov += np.eye(n) * 1e-8
    increments = rng.multivariate_normal(np.zeros(n), cov).astype(np.float32)
    return np.concatenate([[0.0], np.cumsum(increments)]).astype(np.float32)


def _fallback_sbm(T, alpha, rng):
    t = np.arange(1, T, dtype=np.float64)
    var = np.maximum(t ** alpha - (t - 1) ** alpha, 1e-8)
    increments = rng.normal(0.0, np.sqrt(var)).astype(np.float32)
    return np.concatenate([[0.0], np.cumsum(increments)]).astype(np.float32)


def _fallback_ctrw(T, alpha, rng):
    alpha = np.clip(alpha, 0.05, 0.95)
    positions = np.zeros(T, dtype=np.float32)
    current_pos = 0.0
    current_time = 0
    while current_time < T:
        wait = int(max(1, rng.pareto(alpha) + 1))
        jump = rng.normal()
        end_time = min(T, current_time + wait)
        positions[current_time:end_time] = current_pos
        current_pos += jump
        current_time = end_time
    return positions - positions[0]


def _fallback_lw(T, alpha, rng):
    positions = np.zeros(T, dtype=np.float32)
    t = 1
    x = 0.0
    while t < T:
        duration = int(max(1, rng.pareto(max(alpha - 1.0, 0.1)) + 1))
        velocity = rng.choice([-1.0, 1.0])
        for _ in range(duration):
            if t >= T:
                break
            x += velocity
            positions[t] = x
            t += 1
    return positions - positions[0]


def _fallback_attm(T, alpha, rng):
    positions = np.zeros(T, dtype=np.float32)
    t = 1
    x = 0.0
    while t < T:
        duration = int(rng.integers(5, 25))
        D = 10 ** rng.uniform(-2.0, 0.5)
        for _ in range(duration):
            if t >= T:
                break
            x += rng.normal(0, math.sqrt(2 * D))
            positions[t] = x
            t += 1
    return positions - positions[0]


def generate_with_fallback(model_name, T, alpha, rng):
    if model_name == "FBM":
        return _fallback_fbm(T, alpha, rng)
    if model_name == "SBM":
        return _fallback_sbm(T, alpha, rng)
    if model_name == "CTRW":
        return _fallback_ctrw(T, alpha, rng)
    if model_name == "LW":
        return _fallback_lw(T, alpha, rng)
    if model_name == "ATTM":
        return _fallback_attm(T, alpha, rng)
    raise ValueError(model_name)


def generate_model_trajectory(model_name, T, alpha, rng):
    """
    Genera una trayectoria 1D de longitud T.
    """
    try:
        return generate_with_andi(model_name, T, alpha)
    except Exception as exc:
        if ALLOW_FALLBACK_GENERATORS:
            print(f"ADVERTENCIA: usando generador aproximado para {model_name}. Error AnDi: {repr(exc)}")
            return generate_with_fallback(model_name, T, alpha, rng)
        raise

## 5. Construcción de trayectorias con y sin punto de cambio

Para una trayectoria con cambio, se genera:

\[
M_1 \rightarrow M_2, \qquad M_1 \neq M_2.
\]

La posición `cp` se muestrea de forma relativa para que todas las longitudes sean comparables:

\[
u \sim \mathcal{U}(0.2,0.8), \qquad cp = \mathrm{round}(u L).
\]

Para \(L=200\), esto da `cp` entre 40 y 160.  
Además, cada trayectoria se genera con un nivel de ruido `noise_sigma` tomado de:

```python
NOISE_LEVELS = [0.1, 0.5, 1.0]
```

La clase usada para los incrementos es:

\[
cp\_dx = cp - 1.
\]


In [ ]:
def standardize_track(x):
    """
    Estandariza la trayectoria usando la desviación estándar de sus incrementos.
    """
    x = np.asarray(x, dtype=np.float32)
    x = x - x[0]
    dx = np.diff(x)
    scale = float(np.std(dx))
    if scale < 1e-6:
        scale = 1.0
    return (x / scale).astype(np.float32)


def add_localization_noise(x, noise_sigma, rng):
    """
    Añade ruido gaussiano de localización y recentra la trayectoria.
    """
    x = np.asarray(x, dtype=np.float32)
    if noise_sigma is not None and float(noise_sigma) > 0:
        x = x + rng.normal(0.0, float(noise_sigma), size=x.shape).astype(np.float32)
    return (x - x[0]).astype(np.float32)


def sample_noise_sigma(rng):
    """
    Selecciona un nivel de ruido entre los niveles definidos.
    """
    return float(rng.choice(NOISE_LEVELS))


def sample_changepoint(rng):
    """
    Muestrea una posición relativa u en [0.2, 0.8] y la transforma en cp.
    Esto mantiene la distribución relativa de los puntos de cambio al cambiar L.
    """
    u = float(rng.uniform(CP_RELATIVE_MIN, CP_RELATIVE_MAX))
    cp = int(round(u * LENGTH))
    cp = int(np.clip(cp, VALID_CP_MIN, VALID_CP_MAX))
    return cp, cp / LENGTH


def make_changepoint_track(model1, model2, cp, cp_relative_position, noise_sigma, rng):
    """
    Crea una trayectoria de longitud LENGTH con transición model1 -> model2.
    """
    alpha1 = sample_alpha(model1, rng)
    alpha2 = sample_alpha(model2, rng)

    # Primer segmento: cp puntos.
    segment1 = standardize_track(generate_model_trajectory(model1, cp, alpha1, rng))

    # Segundo segmento: necesita L - cp + 1 puntos para conectar continuidad.
    segment2 = standardize_track(generate_model_trajectory(model2, LENGTH - cp + 1, alpha2, rng))

    # Concatenación continua: se elimina el primer punto del segundo segmento.
    x = np.concatenate([segment1, segment1[-1] + segment2[1:]]).astype(np.float32)

    if len(x) != LENGTH:
        raise ValueError(f"Longitud incorrecta: {len(x)} != {LENGTH}")

    x = add_localization_noise(x, noise_sigma, rng)
    dx = np.diff(x).astype(np.float32)

    return {
        "x": x,
        "dx": dx,
        "has_changepoint": 1,
        "cp": int(cp),
        "cp_dx": int(cp - 1),
        "cp_relative_position": float(cp_relative_position),
        "noise_sigma": float(noise_sigma),
        "model1": model1,
        "model2": model2,
        "alpha1": float(alpha1),
        "alpha2": float(alpha2),
    }


def make_no_changepoint_track(model_name, noise_sigma, rng):
    """
    Crea una trayectoria homogénea sin punto de cambio.
    """
    alpha = sample_alpha(model_name, rng)
    x = standardize_track(generate_model_trajectory(model_name, LENGTH, alpha, rng))
    x = add_localization_noise(x, noise_sigma, rng)
    dx = np.diff(x).astype(np.float32)

    return {
        "x": x,
        "dx": dx,
        "has_changepoint": 0,
        "cp": -1,
        "cp_dx": -1,
        "cp_relative_position": np.nan,
        "noise_sigma": float(noise_sigma),
        "model1": model_name,
        "model2": model_name,
        "alpha1": float(alpha),
        "alpha2": float(alpha),
    }


## 6. Plan de equilibrio por split

Cada split se divide en dos partes:

- 50% trayectorias con punto de cambio;
- 50% trayectorias sin punto de cambio.

Las trayectorias con cambio se distribuyen de forma equilibrada entre las 20 transiciones ordenadas.  
Las trayectorias sin cambio se distribuyen de forma equilibrada entre los 5 modelos.

In [ ]:
def allocate_counts(total, labels):
    """
    Reparte total ejemplos entre labels de la forma más equilibrada posible.
    """
    labels = list(labels)
    base = total // len(labels)
    remainder = total % len(labels)
    counts = {label: base for label in labels}
    for label in labels[:remainder]:
        counts[label] += 1
    return counts


def build_split_plan(split_size):
    n_with = split_size // 2
    n_without = split_size - n_with

    transition_counts = allocate_counts(n_with, TRANSITIONS)
    no_cp_counts = allocate_counts(n_without, MODEL_NAMES)

    return transition_counts, no_cp_counts


for split_name, split_size in SPLIT_SIZES.items():
    transition_counts, no_cp_counts = build_split_plan(split_size)
    print(split_name, "total:", split_size)
    print("  with CP:", sum(transition_counts.values()), "per transition min/max:", min(transition_counts.values()), max(transition_counts.values()))
    print("  no CP:", sum(no_cp_counts.values()), "per model:", no_cp_counts)

## 7. Generación y guardado en HDF5

Los ficheros se guardan en:

```text
data_synthetic_with_without_changepoint_dx/
```

con los nombres:

```text
train_L200_dim1_with_without_dx.h5
val_L200_dim1_with_without_dx.h5
test_L200_dim1_with_without_dx.h5
```

In [ ]:
def create_empty_h5(file_path, n_examples):
    string_dtype = h5py.string_dtype(encoding="utf-8")

    with h5py.File(file_path, "w") as f:
        f.create_dataset("x", shape=(n_examples, LENGTH), dtype="float32", chunks=True)
        f.create_dataset("dx", shape=(n_examples, DX_LENGTH), dtype="float32", chunks=True)
        f.create_dataset("has_changepoint", shape=(n_examples,), dtype="int8", chunks=True)
        f.create_dataset("cp", shape=(n_examples,), dtype="int16", chunks=True)
        f.create_dataset("cp_dx", shape=(n_examples,), dtype="int16", chunks=True)
        f.create_dataset("cp_relative_position", shape=(n_examples,), dtype="float32", chunks=True)
        f.create_dataset("noise_sigma", shape=(n_examples,), dtype="float32", chunks=True)
        f.create_dataset("model1", shape=(n_examples,), dtype=string_dtype, chunks=True)
        f.create_dataset("model2", shape=(n_examples,), dtype=string_dtype, chunks=True)
        f.create_dataset("alpha1", shape=(n_examples,), dtype="float32", chunks=True)
        f.create_dataset("alpha2", shape=(n_examples,), dtype="float32", chunks=True)

        f.attrs["created_at"] = datetime.now().isoformat()
        f.attrs["length"] = LENGTH
        f.attrs["dx_length"] = DX_LENGTH
        f.attrs["dimension"] = DIMENSION
        f.attrs["min_segment_length"] = MIN_SEGMENT_LENGTH
        f.attrs["cp_relative_min"] = CP_RELATIVE_MIN
        f.attrs["cp_relative_max"] = CP_RELATIVE_MAX
        f.attrs["valid_cp_min"] = VALID_CP_MIN
        f.attrs["valid_cp_max"] = VALID_CP_MAX
        f.attrs["valid_dx_min"] = VALID_DX_MIN
        f.attrs["valid_dx_max"] = VALID_DX_MAX
        f.attrs["noise_levels"] = json.dumps(NOISE_LEVELS)
        f.attrs["model_names"] = json.dumps(MODEL_NAMES)
        f.attrs["run_mode"] = RUN_MODE
        f.attrs["generator"] = "andi_datasets" if ANDI_MODEL_OBJECT is not None else "fallback_debug"
        f.attrs["andi_datasets_version"] = ANDI_DATASETS_VERSION
        f.attrs["numpy_version"] = np.__version__
        f.attrs["h5py_version"] = h5py.__version__
        f.attrs["comparability_note"] = "L200 uses noise levels and relative changepoint range matching L100 protocol."


def write_record(f, idx, record):
    f["x"][idx] = record["x"]
    f["dx"][idx] = record["dx"]
    f["has_changepoint"][idx] = record["has_changepoint"]
    f["cp"][idx] = record["cp"]
    f["cp_dx"][idx] = record["cp_dx"]
    f["cp_relative_position"][idx] = record["cp_relative_position"]
    f["noise_sigma"][idx] = record["noise_sigma"]
    f["model1"][idx] = record["model1"]
    f["model2"][idx] = record["model2"]
    f["alpha1"][idx] = record["alpha1"]
    f["alpha2"][idx] = record["alpha2"]


def generate_split(split_name, split_size, seed):
    rng = np.random.default_rng(seed)
    file_path = DATA_DIR / f"{split_name}_L{LENGTH}_dim{DIMENSION}_with_without_dx.h5"
    summary_path = DATA_DIR / f"{split_name}_L{LENGTH}_dim{DIMENSION}_with_without_dx_summary.csv"

    if file_path.exists():
        if OVERWRITE_EXISTING:
            print(f"El fichero ya existe y se va a regenerar: {file_path}")
            file_path.unlink()
            if summary_path.exists():
                summary_path.unlink()
        else:
            raise FileExistsError(
                f"El fichero ya existe: {file_path}. "
                "Pon OVERWRITE_EXISTING = True si quieres regenerarlo."
            )

    transition_counts, no_cp_counts = build_split_plan(split_size)
    create_empty_h5(file_path, split_size)

    idx = 0
    summary_rows = []

    with h5py.File(file_path, "a") as f:
        # Trayectorias con punto de cambio.
        for (model1, model2), count in transition_counts.items():
            for _ in range(count):
                cp, cp_relative_position = sample_changepoint(rng)
                noise_sigma = sample_noise_sigma(rng)
                record = make_changepoint_track(model1, model2, cp, cp_relative_position, noise_sigma, rng)
                write_record(f, idx, record)

                summary_rows.append({
                    "split": split_name,
                    "has_changepoint": 1,
                    "model1": model1,
                    "model2": model2,
                    "transition": f"{model1} → {model2}",
                    "cp": cp,
                    "cp_relative_position": cp_relative_position,
                    "noise_sigma": noise_sigma,
                })
                idx += 1

                if idx % 5000 == 0:
                    print(f"{split_name}: {idx}/{split_size}")

        # Trayectorias sin punto de cambio.
        for model_name, count in no_cp_counts.items():
            for _ in range(count):
                noise_sigma = sample_noise_sigma(rng)
                record = make_no_changepoint_track(model_name, noise_sigma, rng)
                write_record(f, idx, record)

                summary_rows.append({
                    "split": split_name,
                    "has_changepoint": 0,
                    "model1": model_name,
                    "model2": model_name,
                    "transition": f"{model_name} (sin cambio)",
                    "cp": -1,
                    "cp_relative_position": np.nan,
                    "noise_sigma": noise_sigma,
                })
                idx += 1

                if idx % 5000 == 0:
                    print(f"{split_name}: {idx}/{split_size}")

        # Aleatorizar el orden dentro del archivo.
        permutation = rng.permutation(split_size)
        for key in [
            "x", "dx", "has_changepoint", "cp", "cp_dx", "cp_relative_position",
            "noise_sigma", "model1", "model2", "alpha1", "alpha2"
        ]:
            data = f[key][:]
            f[key][:] = data[permutation]

    summary = pd.DataFrame(summary_rows).iloc[permutation].reset_index(drop=True)
    summary.to_csv(summary_path, index=False)

    print(f"Guardado: {file_path}")
    print(f"Resumen: {summary_path}")

    return file_path, summary_path


## 8. Ejecutar la generación

Ejecuta esta celda para crear `train`, `val` y `test`.

Si estás en modo `FAST`, se generarán ficheros pequeños.  
Si estás en modo `GLOBAL`, la ejecución puede tardar bastante.

In [ ]:
generated_files = {}

for i, (split_name, split_size) in enumerate(SPLIT_SIZES.items()):
    file_path, summary_path = generate_split(
        split_name=split_name,
        split_size=split_size,
        seed=RANDOM_SEED + 100 * i,
    )
    generated_files[split_name] = {
        "h5": str(file_path),
        "summary": str(summary_path),
    }

generated_files

## 9. Verificación de los ficheros generados

Esta celda comprueba que:

- `dx` tiene longitud 199;
- las etiquetas binarias están equilibradas;
- existen 20 transiciones con punto de cambio;
- existen trayectorias sin cambio para los 5 modelos.

In [ ]:
def decode_array(values):
    decoded = []
    for v in values:
        if isinstance(v, bytes):
            decoded.append(v.decode("utf-8"))
        else:
            decoded.append(str(v))
    return np.array(decoded, dtype=object)


def inspect_h5(split_name):
    file_path = DATA_DIR / f"{split_name}_L{LENGTH}_dim{DIMENSION}_with_without_dx.h5"
    with h5py.File(file_path, "r") as f:
        has_cp = f["has_changepoint"][:]
        cp_values = f["cp"][:]
        noise_sigma = f["noise_sigma"][:] if "noise_sigma" in f else np.full_like(has_cp, np.nan, dtype="float32")

        mask_cp = has_cp == 1
        info = {
            "file": str(file_path),
            "x_shape": f["x"].shape,
            "dx_shape": f["dx"].shape,
            "has_changepoint_counts": dict(zip(*np.unique(has_cp, return_counts=True))),
            "cp_min_with_cp": int(cp_values[mask_cp].min()),
            "cp_max_with_cp": int(cp_values[mask_cp].max()),
            "cp_relative_min_with_cp": float((cp_values[mask_cp] / LENGTH).min()),
            "cp_relative_max_with_cp": float((cp_values[mask_cp] / LENGTH).max()),
            "noise_levels_observed": sorted([float(x) for x in np.unique(noise_sigma)]),
            "attrs_length": int(f.attrs["length"]),
            "attrs_dx_length": int(f.attrs["dx_length"]),
            "attrs_noise_levels": f.attrs.get("noise_levels", "not_available"),
        }

        model1 = decode_array(f["model1"][:])
        model2 = decode_array(f["model2"][:])

        transitions = pd.Series(
            [f"{m1} → {m2}" for m1, m2 in zip(model1[mask_cp], model2[mask_cp])]
        )
        no_cp_models = pd.Series(model1[has_cp == 0])
        noise_counts = pd.Series(noise_sigma).value_counts().sort_index()

        transition_counts = transitions.value_counts().sort_index()
        no_cp_counts = no_cp_models.value_counts().sort_index()

    return info, transition_counts, no_cp_counts, noise_counts


all_infos = []
for split_name in SPLIT_SIZES:
    info, transition_counts, no_cp_counts, noise_counts = inspect_h5(split_name)
    all_infos.append(info)

    print("\n", "=" * 80)
    print(split_name)
    print(info)
    print("\nTransiciones con cambio:")
    print(transition_counts)
    print("\nModelos sin cambio:")
    print(no_cp_counts)
    print("\nNiveles de ruido:")
    print(noise_counts)

pd.DataFrame(all_infos)


## 10. Visualización rápida de ejemplos

Esta celda dibuja algunas trayectorias para comprobar visualmente que el fichero se ha generado correctamente.

In [ ]:
def plot_examples(split_name="train", n_examples=6, seed=42):
    rng = np.random.default_rng(seed)
    file_path = DATA_DIR / f"{split_name}_L{LENGTH}_dim{DIMENSION}_with_without_dx.h5"

    with h5py.File(file_path, "r") as f:
        total = len(f["x"])
        indices = rng.choice(total, size=min(n_examples, total), replace=False)

        plt.figure(figsize=(12, 8))
        for i, idx in enumerate(indices, start=1):
            x = f["x"][idx]
            has_cp = int(f["has_changepoint"][idx])
            cp = int(f["cp"][idx])
            m1 = f["model1"][idx].decode("utf-8") if isinstance(f["model1"][idx], bytes) else str(f["model1"][idx])
            m2 = f["model2"][idx].decode("utf-8") if isinstance(f["model2"][idx], bytes) else str(f["model2"][idx])

            ax = plt.subplot(math.ceil(n_examples / 2), 2, i)
            ax.plot(x, linewidth=1)
            if has_cp == 1:
                ax.axvline(cp, linestyle="--")
                ax.set_title(f"{m1} → {m2}, cp={cp}")
            else:
                ax.set_title(f"{m1}, sin cambio")
            ax.set_xlabel("t")
            ax.set_ylabel("x(t)")

        plt.tight_layout()
        plt.show()

plot_examples("train", n_examples=6)

## 11. Resultado esperado para el notebook ConvTransformer \(L=200\)

Después de ejecutar este cuaderno, el notebook de entrenamiento debe cargar:

```python
train_L200_dim1_with_without_dx.h5
val_L200_dim1_with_without_dx.h5
test_L200_dim1_with_without_dx.h5
```

y debe mostrar:

```text
dx_shape = (199, 1)
```

Si aparece `dx_shape = (99, 1)`, significa que se están cargando todavía los datos de \(L=100\).